# Joint last-layer Laplace for source-directed exploration

This example uses `GasSourceEstimator` for **known wind** and its `JointSourceLaplace` component. Both concentration and source networks are trained jointly; their hidden layers and the wind are treated as fixed during the Laplace calculation.

1. **Fixed-data stability:** check the concentration/source estimate under continued training with the same observations.
2. **Sequential acquisition:** add measurements using the expected reduction of last-layer **source-parameter variance**, and compare with random sampling, geodesic coverage, and training without new measurements.

The score is a proxy for source-location accuracy, which is evaluated separately. The experiment selects measurement locations; it does not yet plan robot motion or account for travel costs.

In [ ]:
from pathlib import Path
import copy
import heapq
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.interpolate import NearestNDInterpolator

root = Path.cwd()
if not (root / "source_estimation_pinn").exists():
    root = root.parent
sys.path.insert(0, str(root))

from source_estimation_pinn.environment import OccupancyGrid, WindField
from source_estimation_pinn.measurements import GasSampleSet
from source_estimation_pinn.estimation import GasSourceConfig, GasSourceEstimator

## Data and structured detection route

Before the first meaningful gas detection, the stationary inverse problem admits the trivial solution $c\approx0$, $q\approx0$. A source-directed uncertainty method cannot infer a source from only zero measurements.

Therefore this experiment assumes a coverage phase has already produced a short structured route containing several detections. The route still contains only a small subset of the full reference field. The complete CSV is used solely as a noise-free sensor oracle and for evaluation.

In [ ]:
mesh_path = root / "data/example_labyrinth/labyrinth_2d_fine.msh"
wind_path = root / "data/example_labyrinth/wind_gt.csv"
gas_path = root / "data/example_labyrinth/gas_gt1.5_2.0.csv"
true_source = np.array([1.5, 2.0])

occupancy = OccupancyGrid.from_msh_file(mesh_path, resolution=0.1)
wind = WindField.from_csv(wind_path)
gas_reference = GasSampleSet.from_csv(gas_path)
sensor_oracle = NearestNDInterpolator(
    gas_reference.positions,
    gas_reference.concentrations,
)

candidate_points = occupancy.free_points
reference_concentration = np.asarray(sensor_oracle(candidate_points))

# Coverage route followed until it contains several nonzero detections.
route_targets = np.vstack((
    np.column_stack((np.linspace(4, 6, 6), np.full(6, 1.5))),
    np.column_stack((np.full(6, 5.0), np.linspace(0.5, 2.5, 6))),
))
squared_distances = np.sum(
    (route_targets[:, None, :] - gas_reference.positions[None, :, :]) ** 2,
    axis=2,
)
nearest_indices = np.argmin(squared_distances, axis=1)
_, first_occurrence = np.unique(nearest_indices, return_index=True)
nearest_indices = nearest_indices[np.sort(first_occurrence)]
initial_samples = GasSampleSet(
    positions=gas_reference.positions[nearest_indices],
    concentrations=gas_reference.concentrations[nearest_indices],
)

detection_threshold = 1e-3
print(f"Initial measurements: {len(initial_samples.positions)}")
print(f"Detections above {detection_threshold:g}: "
      f"{np.count_nonzero(initial_samples.concentrations > detection_threshold)}")
print(f"Maximum measured concentration: "
      f"{initial_samples.concentrations.max():.4e}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
occupancy.plot(ax=ax, title="Structured route after first gas detections")
route_plot = ax.scatter(
    initial_samples.positions[:, 0], initial_samples.positions[:, 1],
    c=initial_samples.concentrations, cmap="plasma", s=35,
    edgecolor="white", linewidth=0.5, label="gas measurements", zorder=3,
)
ax.plot(initial_samples.positions[:, 0], initial_samples.positions[:, 1],
        color="white", linewidth=0.8, zorder=2)
ax.scatter(*true_source, marker="*", s=130, color="lime",
           edgecolor="black", label="true source", zorder=4)
fig.colorbar(route_plot, ax=ax, label="measured concentration")
ax.legend()
plt.show()

## Phase 1: stability of the fixed-data MAP estimate

We repeatedly continue training with exactly the same measurements. A usable Laplace approximation requires a meaningful local optimum. The diagnostic records:

- fit at the actual measurements;
- full-field error, used only because simulation ground truth is available;
- spatial standard deviation of $c_\phi$ to detect collapse to a constant field;
- source-location error.

The Laplace phase is stopped automatically if the final field is non-finite, effectively constant, or fits the measurements worse than the first checkpoint.

In [ ]:
config = GasSourceConfig(
    concentration_hidden_layers=3,
    concentration_hidden_dim=32,
    source_hidden_layers=3,
    source_hidden_dim=16,
    diffusion_constant=1e-3,
    lambda_sparse=1e-4,
    lambda_nonfree=1e-2,
    # The earlier value 1e-3 noticeably distorted the tiny gas-loss scale.
    last_layer_prior_precision=1e-6,
    learning_rate=2e-3,
)

model_seed = 7
checkpoint_steps = 300
n_checkpoints = 6
n_collocation_points = min(500, len(candidate_points))

torch.manual_seed(model_seed)
np.random.seed(model_seed)
stable_estimator = GasSourceEstimator(
    occupancy=occupancy,
    wind=wind,
    config=config,
)

stability = {
    name: [] for name in (
        "measurement_rmse", "field_rmse", "spatial_std", "source_error"
    )
}
concentration_snapshots = []
source_snapshots = []

for checkpoint in range(n_checkpoints):
    stable_estimator.fit(
        initial_samples,
        steps=checkpoint_steps,
        n_collocation_points=n_collocation_points,
    )
    c_field = stable_estimator.predict_concentration(candidate_points)
    q_field = stable_estimator.predict_source(candidate_points)
    c_measured = stable_estimator.predict_concentration(
        initial_samples.positions
    )
    stability["measurement_rmse"].append(np.sqrt(np.mean(
        (c_measured - initial_samples.concentrations) ** 2
    )))
    stability["field_rmse"].append(np.sqrt(np.mean(
        (c_field - reference_concentration) ** 2
    )))
    stability["spatial_std"].append(np.std(c_field))
    stability["source_error"].append(np.linalg.norm(
        stable_estimator.get_source_estimate() - true_source
    ))
    concentration_snapshots.append(c_field)
    source_snapshots.append(q_field)

for values in stability.values():
    values[:] = np.asarray(values)

map_is_stable = (
    all(np.all(np.isfinite(values)) for values in stability.values())
    and stability["spatial_std"][-1] > 1e-5
    and stability["measurement_rmse"][-1]
        <= 1.1 * stability["measurement_rmse"][0]
)
print(f"Stable nonconstant MAP estimate: {map_is_stable}")
print(f"Final measurement RMSE: {stability['measurement_rmse'][-1]:.4e}")
print(f"Final concentration spatial std: {stability['spatial_std'][-1]:.4e}")
print(f"Final source error: {stability['source_error'][-1]:.3f} m")
if not map_is_stable:
    raise RuntimeError("MAP stability gate failed; do not compute Laplace approximation")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
steps = checkpoint_steps * np.arange(1, n_checkpoints + 1)
for ax, name, ylabel in (
    (axes[0, 0], "measurement_rmse", "measurement RMSE"),
    (axes[0, 1], "field_rmse", "full-field RMSE"),
    (axes[1, 0], "spatial_std", "spatial std of concentration"),
    (axes[1, 1], "source_error", "source-location error (m)"),
):
    ax.plot(steps, stability[name], "o-")
    ax.set(xlabel="optimizer step", ylabel=ylabel, title=name.replace("_", " "))
plt.show()

## Joint final-layer posterior through the framework

The estimator supplies `loss_for_last_layer(theta)` using its actual gas/source losses, fixed training data, and configured prior on both final layers. `estimator.laplace` owns the Hessian calculation and cached covariance, with parameters ordered as concentration weights/bias followed by source weights/bias:

$$
\theta=\begin{bmatrix}\theta_c\\\theta_q\end{bmatrix},\qquad
\Sigma\approx\left(\nabla_\theta^2L(\theta^\ast)\right)^{-1}
=\begin{bmatrix}\Sigma_{cc}&\Sigma_{cq}\\\Sigma_{qc}&\Sigma_{qq}\end{bmatrix}.
$$

`fit()` and `training_step()` invalidate the cache. The next score evaluation then recomputes it for the updated model.

As in the original MWE, the framework symmetrizes the Hessian and floors its eigenvalues at `1e-7` before inversion. The eigendecomposition and covariance use `float64` because reconstructing the ill-conditioned covariance in `float32` can destroy its positive definiteness. This yields the inverse of a stabilized Hessian, not necessarily the original Hessian. The check below still stops acquisition if the result is invalid.

In [ ]:
covariance = stable_estimator.laplace.compute_covariance()
print("Joint last-layer covariance shape:", tuple(covariance.shape))

## Expected reduction of source-parameter uncertainty

For a noise-free concentration observation at candidate $x$, the linearized Gaussian approximation gives

$$
s(x)=\frac{\|\Sigma_{qc}\nabla_{\theta_c}c\|_2^2}
{\nabla_{\theta_c}c^\top\Sigma_{cc}\nabla_{\theta_c}c},
\qquad x_{\mathrm{next}}=\arg\max_x s(x).
$$

The framework computes $\nabla_{\theta_c}c=\operatorname{sigmoid}(z_c^\ast)\widetilde h_c$ and evaluates this score. Its current implementation adds `1e-8` to the denominator for numerical stabilization; this is not a modeled sensor-noise variance.

The previous notebook scored integrated **source-field** variance, including source-network output gradients and sensor noise. The current API scores **source-parameter** variance, as derived in [the mathematical explanation](../docs/joint_last_layer_laplace.md). Its results therefore need not match the previous experiment.

The helper below only validates the covariance and batches API calls to limit memory use. All Laplace and score calculations live in the framework.

In [ ]:
def acquisition_scores(estimator, points, batch_size=512):
    covariance = estimator.laplace.covariance
    if covariance is None:
        covariance = estimator.laplace.compute_covariance()
    if not torch.isfinite(covariance).all():
        raise RuntimeError("Laplace covariance is non-finite.")
    symmetric_covariance = 0.5 * (covariance + covariance.T)
    _, info = torch.linalg.cholesky_ex(symmetric_covariance)
    if info.item() != 0:
        raise RuntimeError(
            "Joint Laplace covariance is not positive definite. "
            "Check the trained optimum and Hessian before selecting measurements; "
            "the covariance remains invalid despite eigenvalue clipping."
        )

    with torch.no_grad():
        scores = torch.cat([
            estimator.source_uncertainty_reduction(points[start:start + batch_size])
            for start in range(0, len(points), batch_size)
        ]).cpu().numpy()
    if not np.all(np.isfinite(scores)) or np.any(scores < 0):
        raise RuntimeError("Expected finite, nonnegative acquisition scores.")
    return scores


initial_information_score = acquisition_scores(stable_estimator, candidate_points)
print("Highest-scoring candidate:",
      candidate_points[np.argmax(initial_information_score)])

## Phase 2: sequential acquisition comparison

All strategies start from an exact deep copy of the stable estimator, including Adam state. Each round gives every strategy the same number of additional optimizer steps.

Compared strategies:

- **joint Laplace:** expected reduction of last-layer source-parameter variance;
- **random:** random unmeasured free cell;
- **geodesic coverage:** farthest collision-free distance from existing measurements;
- **training only:** no new measurement, same additional training.

In [ ]:
def geodesic_distance_to_measurements(grid, measurement_positions):
    traversable = grid.free_mask
    height, width = traversable.shape
    distances = np.full((height, width), np.inf)
    queue = []
    rows, cols = np.nonzero(traversable)
    centers = np.column_stack((grid.x_centers[cols], grid.y_centers[rows]))

    for position in measurement_positions:
        nearest = np.argmin(np.sum((centers - position) ** 2, axis=1))
        row, col = rows[nearest], cols[nearest]
        distances[row, col] = 0.0
        heapq.heappush(queue, (0.0, row, col))

    moves = (
        (-1, 0, 1.0), (1, 0, 1.0), (0, -1, 1.0), (0, 1, 1.0),
        (-1, -1, np.sqrt(2.0)), (-1, 1, np.sqrt(2.0)),
        (1, -1, np.sqrt(2.0)), (1, 1, np.sqrt(2.0)),
    )
    while queue:
        distance, row, col = heapq.heappop(queue)
        if distance > distances[row, col]:
            continue
        for drow, dcol, length in moves:
            next_row, next_col = row + drow, col + dcol
            if not (0 <= next_row < height and 0 <= next_col < width):
                continue
            if not traversable[next_row, next_col]:
                continue
            if drow and dcol and not (
                traversable[row + drow, col]
                and traversable[row, col + dcol]
            ):
                continue
            candidate = distance + length * grid.resolution
            if candidate < distances[next_row, next_col]:
                distances[next_row, next_col] = candidate
                heapq.heappush(queue, (candidate, next_row, next_col))
    return distances[grid.free_mask]


def nearest_candidate_indices(positions):
    return {
        int(np.argmin(np.sum((candidate_points - position) ** 2, axis=1)))
        for position in positions
    }


def evaluate(estimator):
    concentration = estimator.predict_concentration(candidate_points)
    return (
        np.sqrt(np.mean((concentration - reference_concentration) ** 2)),
        np.linalg.norm(estimator.get_source_estimate() - true_source),
    )


strategies = (
    "joint c-q Laplace", "random", "geodesic coverage", "training only"
)
n_acquisitions = 15
update_training_steps = 300
random_seed = 23
results = {}

for strategy in strategies:
    estimator = copy.deepcopy(stable_estimator)
    samples = GasSampleSet(
        initial_samples.positions.copy(),
        initial_samples.concentrations.copy(),
    )
    selected = nearest_candidate_indices(samples.positions)
    acquired_positions = []
    selected_scores = []
    rng = np.random.default_rng(random_seed)

    field_rmse, source_error = evaluate(estimator)
    field_history = [field_rmse]
    source_history = [source_error]

    for acquisition in range(n_acquisitions):
        if strategy != "training only":
            available = np.ones(len(candidate_points), dtype=bool)
            available[list(selected)] = False

            if strategy == "joint c-q Laplace":
                scores = acquisition_scores(estimator, candidate_points)
                index = int(np.argmax(np.where(available, scores, -np.inf)))
                selected_scores.append(scores.copy())
            elif strategy == "random":
                index = int(rng.choice(np.flatnonzero(available)))
            else:
                distances = geodesic_distance_to_measurements(
                    occupancy, samples.positions
                )
                selectable = available & np.isfinite(distances)
                index = int(np.argmax(np.where(
                    selectable, distances, -np.inf
                )))

            position = candidate_points[index]
            concentration = float(
                np.asarray(sensor_oracle(position)).reshape(-1)[0]
            )
            samples.append(position, concentration)
            selected.add(index)
            acquired_positions.append(position.copy())

        estimator.fit(
            samples,
            steps=update_training_steps,
            n_collocation_points=n_collocation_points,
        )
        field_rmse, source_error = evaluate(estimator)
        field_history.append(field_rmse)
        source_history.append(source_error)

    results[strategy] = {
        "estimator": estimator,
        "samples": samples,
        "acquired_positions": np.asarray(acquired_positions).reshape(-1, 2),
        "information_scores": selected_scores,
        "field_rmse": np.asarray(field_history),
        "source_error": np.asarray(source_history),
    }
    print(
        f"{strategy:21s}: concentration RMSE={field_rmse:.4e}, "
        f"source error={source_error:.3f} m"
    )

## Acquisition behavior and estimation quality

First compare every acquisition strategy with **training only**. Only improvements beyond that curve can be attributed to new measurements. Then compare joint Laplace with random and coverage.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)
for ax, strategy in zip(axes.flat, strategies):
    occupancy.plot(ax=ax, title=strategy)
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()
    ax.plot(initial_samples.positions[:, 0], initial_samples.positions[:, 1],
            "w.-", linewidth=0.8, markersize=3,
            label="initial route", zorder=3)
    added = results[strategy]["acquired_positions"]
    if len(added):
        scatter = ax.scatter(
            added[:, 0], added[:, 1],
            c=np.arange(1, len(added) + 1), cmap="viridis",
            vmin=1, vmax=n_acquisitions, s=55,
            edgecolor="black", zorder=4,
        )
        for number, position in enumerate(added, start=1):
            ax.annotate(str(number), position, xytext=(4, 4),
                        textcoords="offset points", color="white", zorder=5)
        fig.colorbar(scatter, ax=ax, label="acquisition number")
    else:
        ax.text(0.5, 0.08, "no additional measurements",
                transform=ax.transAxes, ha="center",
                bbox={"facecolor": "white", "alpha": 0.8})
    ax.scatter(*true_source, marker="*", s=120, color="lime",
               edgecolor="black", label="true source", zorder=5)
    ax.legend(loc="upper right")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
rounds = np.arange(n_acquisitions + 1)
for strategy in strategies:
    axes[0].plot(rounds, results[strategy]["field_rmse"], "o-", label=strategy)
    axes[1].plot(rounds, results[strategy]["source_error"], "o-", label=strategy)
axes[0].set(xlabel="acquisition round", ylabel="full-field concentration RMSE",
            title="Concentration reconstruction")
axes[1].set(xlabel="acquisition round", ylabel="source-location error (m)",
            title="Source localization")
for ax in axes:
    ax.legend()
plt.show()

## What did joint Laplace optimize?

The first map shows the initial expected reduction of last-layer source-parameter variance. The second shows the final learned source field of the joint-Laplace strategy. This makes it possible to check whether selected measurements follow the source-information objective rather than merely spatial distance.

In [ ]:
def as_grid(values):
    field = np.full(occupancy.occupancy.shape, np.nan)
    field[occupancy.free_mask] = values
    return field

extent = [
    occupancy.lower_bound[0], occupancy.upper_bound[0],
    occupancy.lower_bound[1], occupancy.upper_bound[1],
]
joint_result = results["joint c-q Laplace"]
final_estimator = joint_result["estimator"]
final_source = final_estimator.predict_source(candidate_points)
final_concentration = final_estimator.predict_concentration(candidate_points)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
for ax, values, title, label, cmap in (
    (axes[0], initial_information_score,
     "Initial source-parameter variance reduction", "information score", "viridis"),
    (axes[1], final_source,
     "Final learned source field", "q", "plasma"),
    (axes[2], final_concentration,
     "Final learned concentration", "concentration", "plasma"),
):
    occupancy.plot(ax=ax, title=title)
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()
    image = ax.imshow(as_grid(values), origin="lower", extent=extent,
                      interpolation="nearest", cmap=cmap, alpha=0.9, zorder=2)
    ax.scatter(*true_source, marker="*", s=110, color="lime",
               edgecolor="black", label="true source", zorder=4)
    fig.colorbar(image, ax=ax, label=label)
axes[1].scatter(*final_estimator.get_source_estimate(), marker="x", s=70,
                color="cyan", label="estimated source", zorder=5)
axes[1].legend()
plt.show()

## Interpretation

This notebook enforces the correct order of questions:

1. **Is there informative gas data?** Without a detection, use coverage rather than source-directed inference.
2. **Is the fixed-data MAP estimate stable and nonconstant?** If not, Laplace around that point is meaningless.
3. **Do added measurements outperform training-only?** Otherwise apparent progress is only extra optimization.
4. **Does joint $c$–$q$ Laplace outperform random and geodesic coverage?** Only then does its source-directed covariance provide useful acquisition information.

This is still a single local Gaussian approximation. It cannot represent multiple widely separated source modes, and the sum of last-layer source-parameter variances is only a proxy for source-location uncertainty. Repeated seeds are appropriate only after this controlled run behaves sensibly.